<a href="https://colab.research.google.com/github/Chathuranga630/sionna/blob/main/Resilience_Paper_Simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install "sionna>=2.1,<2.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 673.2/673.2 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 271.7/271.7 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 106.3 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found ex

In [ ]:
pip install numpy matplotlib

In [ ]:
import csv
import math
import platform
import sys
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch

try:
    import sionna
    import sionna.phy
    from sionna.phy.channel import gen_single_sector_topology
    from sionna.phy.channel.tr38901 import PanelArray, UMi
except ImportError as e:
    raise SystemExit(
        "\nSionna is not installed.\n"
        "Install the current Sionna release with:\n"
        "    pip install sionna\n"
        "Then restart Python/Jupyter and run this script again.\n"
    ) from e


# ================================================================
# 1. EXPERIMENT CONFIGURATION
# ================================================================

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Sionna 2.x uses PyTorch
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

OUT = Path("r_service_results")
OUT.mkdir(parents=True, exist_ok=True)

# 5G / channel parameters
CARRIER_FREQUENCY_HZ = 3.5e9
SCS_HZ = 30e3
PRB_BANDWIDTH_HZ = 12.0 * SCS_HZ
SLOT_DURATION_S = 1e-3

NUM_USERS = 6
TOTAL_PRBS = 24

# Users 0-2: stringent/high priority
# Users 3-5: relaxed/low priority
HIGH_PRIORITY = np.array([0, 1, 2])
LOW_PRIORITY = np.array([3, 4, 5])

# Expected service rates from the simulation design.
# These are assumptions, NOT values supplied by the paper.
TARGET_MBPS = np.array(
    [12.0, 12.0, 12.0, 5.0, 5.0, 5.0],
    dtype=np.float64
)

DELTA_Q_MBPS = 1.0

# Incident thresholds.
# These are simulation assumptions because the paper does not give
# numerical values for these thresholds.
T_NMIN_MS = 25.0
D_MIN_MS = 8.0
E_TH_MBPS = 4.0
T_PROC_MS = 7.0
T_SLEEP_MAX_MS = 5.0

# Constant delay c in t_resp,i = d_i + c
C_MS = 0.5

# Maximum recovery iterations
MAX_RECOVERY_ITERATIONS = 12

# Power/noise assumptions for the system-level rate mapping
TX_POWER_DBM = 23.0
NOISE_DENSITY_DBM_HZ = -174.0

# ================================================================
# 2. HELPER FUNCTIONS
# ================================================================

def print_header(title):
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)


def save_rows_csv(filename, rows):
    """Save list of dictionaries to CSV."""
    if not rows:
        return

    path = OUT / filename
    keys = list(rows[0].keys())

    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(keys)

        for row in rows:
            values = []
            for key in keys:
                value = row[key]

                if isinstance(value, np.ndarray):
                    value = ";".join(
                        f"{x:.8g}" for x in value.reshape(-1)
                    )

                values.append(value)

            writer.writerow(values)


def save_summary(rows):
    with open(OUT / "summary.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["metric", "value"])

        for metric, value in rows:
            writer.writerow([metric, value])


# ================================================================
# 3. SIONNA 3GPP TR 38.901 UMi CHANNEL
# ================================================================

def build_umi_channel():
    """
    Build a Sionna 3GPP TR 38.901 UMi system-level channel.

    The topology contains one BS and NUM_USERS UEs.
    """

    # 2x2 dual-polarized BS array
    bs_array = PanelArray(
        num_rows_per_panel=2,
        num_cols_per_panel=2,
        polarization="dual",
        polarization_type="cross",
        antenna_pattern="38.901",
        carrier_frequency=CARRIER_FREQUENCY_HZ,
        device=DEVICE,
    )

    # Single-antenna UE
    ut_array = PanelArray(
        num_rows_per_panel=1,
        num_cols_per_panel=1,
        polarization="single",
        polarization_type="V",
        antenna_pattern="omni",
        carrier_frequency=CARRIER_FREQUENCY_HZ,
        device=DEVICE,
    )

    channel_model = UMi(
        carrier_frequency=CARRIER_FREQUENCY_HZ,
        o2i_model="low",
        ut_array=ut_array,
        bs_array=bs_array,
        direction="uplink",
        enable_pathloss=True,
        enable_shadow_fading=True,
        device=DEVICE,
        spec_version="19.2",
    )

    # One fixed topology is used for the complete experiment so that
    # recovery iterations compare resource allocation under the same
    # channel realization.
    topology = gen_single_sector_topology(
        batch_size=1,
        num_ut=NUM_USERS,
        scenario="umi",
        min_ut_velocity=0.0,
        max_ut_velocity=0.0,
        device=DEVICE,
    )

    channel_model.set_topology(*topology)

    return channel_model, topology


def sample_channel_gains(channel_model):
    """
    Sample one Sionna UMi channel realization and reduce it to one
    effective average power gain per UE.

    Sionna 2.1.x channel output is:
       [batch, num_rx, num_rx_ant, num_tx, num_tx_ant, num_paths, time]

    For uplink, the num_tx dimension corresponds to the NUM_USERS UEs.
    The code averages over receiver/link, antenna, path and time dimensions
    while retaining the UE (num_tx) dimension.
    """

    h, tau = channel_model(
        num_time_samples=1,
        sampling_frequency=1e6,
    )

    power = torch.abs(h) ** 2

    # Sionna 2.1.x system-level channel output is:
    # [batch, num_rx, num_rx_ant, num_tx, num_tx_ant, num_paths, num_time_samples].
    # With direction="uplink", the TX links correspond to the UEs and
    # num_tx should therefore equal NUM_USERS.
    #
    # IMPORTANT: Do NOT average the last dimensions in a loop here.
    # The previous implementation collapsed the num_tx (UE) dimension
    # together with antenna/path/time dimensions, which produced a shape
    # such as (1, 1, 8) where 8 was the BS antenna count instead of 6 UEs.
    #
    # We retain the UE dimension (axis 3) and average over:
    #   axis 1 = BS/receiver link dimension
    #   axis 2 = BS/receiver antenna dimension
    #   axis 4 = UE/transmitter antenna dimension
    #   axis 5 = multipath dimension
    #   axis 6 = time dimension
    if power.ndim != 7:
        raise RuntimeError(
            "Unexpected Sionna channel tensor shape. Expected 7 dimensions "
            "[batch, num_rx, num_rx_ant, num_tx, num_tx_ant, num_paths, time], "
            f"got {tuple(power.shape)}"
        )

    user_power = power.mean(dim=(1, 2, 4, 5, 6))

    # user_power = [batch, num_tx]
    if user_power.ndim != 2:
        raise RuntimeError(
            f"Unexpected reduced channel shape: {tuple(user_power.shape)}"
        )

    user_gain = user_power[0]
    user_gain = user_gain.detach().cpu().numpy().astype(np.float64)

    if user_gain.size != NUM_USERS:
        raise RuntimeError(
            "Could not obtain one channel gain per UE. "
            f"Expected {NUM_USERS}, got {user_gain.size}. "
            f"Reduced channel shape: {tuple(power.shape)}"
        )

    user_gain = np.maximum(user_gain, 1e-18)

    return user_gain


# ================================================================
# 4. RATE MODEL
# ================================================================

def rate_from_sionna_gain(gains, prbs, snr_offset_db=0.0):
    """
    Convert Sionna channel power gain + allocated PRBs into a
    system-level achievable-rate estimate.

    This is NOT a decoded PUSCH throughput measurement.

    P_rx = gain * P_PRB
    SNR  = P_rx / N
    rate = PRBs * 180 kHz * log2(1 + SNR)
    """

    gains = np.asarray(gains, dtype=np.float64)
    prbs = np.asarray(prbs, dtype=np.float64)

    total_tx_power_w = 10.0 ** ((TX_POWER_DBM - 30.0) / 10.0)

    # Equal average power per PRB
    power_per_prb_w = total_tx_power_w / TOTAL_PRBS

    noise_dbm = (
        NOISE_DENSITY_DBM_HZ
        + 10.0 * np.log10(PRB_BANDWIDTH_HZ)
        + snr_offset_db
    )

    noise_w = 10.0 ** ((noise_dbm - 30.0) / 10.0)

    snr_linear = (
        gains * power_per_prb_w / noise_w
    )

    snr_linear = np.maximum(snr_linear, 1e-12)

    spectral_efficiency = np.log2(1.0 + snr_linear)

    rates_mbps = (
        prbs
        * PRB_BANDWIDTH_HZ
        * spectral_efficiency
        / 1e6
    )

    return rates_mbps


# ================================================================
# 5. RESOURCE ALLOCATION
# ================================================================

def initial_priority_allocation():
    """
    Initial allocation:
    high-priority users have weight 3,
    low-priority users have weight 1.
    """

    weights = np.ones(NUM_USERS)

    weights[HIGH_PRIORITY] = 3.0
    weights[LOW_PRIORITY] = 1.0

    raw = TOTAL_PRBS * weights / np.sum(weights)

    allocation = np.floor(raw).astype(int)

    # Distribute remaining PRBs according to the largest fractional part.
    while allocation.sum() < TOTAL_PRBS:
        idx = int(np.argmax(raw - allocation))
        allocation[idx] += 1

    # Safety correction
    while allocation.sum() > TOTAL_PRBS:
        candidates = np.where(allocation > 1)[0]
        if len(candidates) == 0:
            break

        idx = int(candidates[np.argmax(allocation[candidates])])
        allocation[idx] -= 1

    return allocation


def recovery_allocation(
    gains,
    target_rates,
    max_iterations=MAX_RECOVERY_ITERATIONS,
    degradation_factor=1.0,
    delay_scale=1.0,
):
    """
    Iteratively reallocate PRBs to restore target QoS.

    degradation_factor < 1 represents a service degradation.

    delay_scale > 1 represents a stressed control/LLM-xApp
    response environment.
    """

    prbs = initial_priority_allocation()

    # Apply degradation by making the effective channel worse.
    effective_gains = gains * degradation_factor

    history = []

    cumulative_talloc_ms = 0.0
    cumulative_d_ms = 0.0

    for iteration in range(1, max_iterations + 1):

        rates = rate_from_sionna_gain(
            effective_gains,
            prbs,
        )

        error = rates - target_rates

        mse = float(np.mean(error ** 2))

        max_abs_error = float(np.max(np.abs(error)))

        min_rate = float(np.min(rates))

        # --------------------------------------------------------
        # Control timing assumptions
        # --------------------------------------------------------
        #
        # These are explicit simulation assumptions, not Sionna
        # PHY timing outputs.
        #

        t_gen_ms = (
            1.5
            + 0.25 * math.log1p(iteration)
            + 0.08 * math.sqrt(max(mse, 0.0))
        )

        d_ms = (
            1.0
            + 0.20 * iteration
            + 0.05 * float(np.std(error))
        )

        t_gen_ms *= delay_scale
        d_ms *= delay_scale

        t_resp_ms = d_ms + C_MS

        cumulative_talloc_ms += (
            t_gen_ms + t_resp_ms
        )

        cumulative_d_ms += d_ms

        # --------------------------------------------------------
        # Incident conditions
        # --------------------------------------------------------

        I1 = int(
            cumulative_talloc_ms > T_NMIN_MS
        )

        I2 = int(
            cumulative_d_ms > D_MIN_MS
        )

        I3 = int(
            max_abs_error > E_TH_MBPS
        )

        history.append(
            {
                "iteration": iteration,
                "prbs": prbs.copy(),
                "rates": rates.copy(),
                "error": error.copy(),
                "mse": mse,
                "min_rate_mbps": min_rate,
                "max_abs_error_mbps": max_abs_error,
                "tgen_ms": t_gen_ms,
                "d_ms": d_ms,
                "tresp_ms": t_resp_ms,
                "talloc_ms": cumulative_talloc_ms,
                "d_cum_ms": cumulative_d_ms,
                "I1": I1,
                "I2": I2,
                "I3": I3,
            }
        )

        # --------------------------------------------------------
        # Recovery stopping criterion
        # --------------------------------------------------------

        required_rate = target_rates - DELTA_Q_MBPS

        if np.all(rates >= required_rate):
            break

        # --------------------------------------------------------
        # Find the most underserved user
        # --------------------------------------------------------

        deficit = np.maximum(
            target_rates - rates,
            0.0,
        )

        receiver = int(
            np.argmax(deficit)
        )

        # Users with more than one PRB can donate one PRB.
        donors = [
            i for i in range(NUM_USERS)
            if i != receiver and prbs[i] > 1
        ]

        if not donors:
            break

        surplus = rates - target_rates

        # Prefer a user with positive surplus.
        positive_surplus = [
            i for i in donors if surplus[i] > 0
        ]

        if positive_surplus:
            donor = max(
                positive_surplus,
                key=lambda i: surplus[i]
            )
        else:
            # If nobody is above target, take from the user
            # with the highest achieved rate.
            donor = max(
                donors,
                key=lambda i: rates[i]
            )

        prbs[donor] -= 1
        prbs[receiver] += 1

    return history


# ================================================================
# 6. SCENARIO EXECUTION
# ================================================================

def run_scenarios(gains):
    """
    Run several service scenarios using the same Sionna channel
    realization.

    This makes the scenarios directly comparable.
    """

    scenarios = [
        {
            "name": "Normal",
            "degradation": 1.00,
            "delay_scale": 1.00,
        },
        {
            "name": "Moderate degradation",
            "degradation": 0.75,
            "delay_scale": 1.00,
        },
        {
            "name": "Severe degradation",
            "degradation": 0.55,
            "delay_scale": 1.00,
        },
        {
            "name": "Severe degradation + control delay",
            "degradation": 0.55,
            "delay_scale": 1.80,
        },
    ]

    all_scenario_rows = []
    histories = {}

    for scenario in scenarios:

        print(
            f"\nRunning scenario: {scenario['name']}"
        )

        history = recovery_allocation(
            gains=gains,
            target_rates=TARGET_MBPS,
            degradation_factor=scenario["degradation"],
            delay_scale=scenario["delay_scale"],
        )

        histories[scenario["name"]] = history

        final = history[-1]

        final_min_rate = float(
            np.min(final["rates"])
        )

        final_mse = float(
            final["mse"]
        )

        final_talloc = float(
            final["talloc_ms"]
        )

        final_dcum = float(
            final["d_cum_ms"]
        )

        all_scenario_rows.append(
            {
                "scenario": scenario["name"],
                "degradation_factor": scenario["degradation"],
                "delay_scale": scenario["delay_scale"],
                "recovery_iterations": len(history),
                "final_min_rate_mbps": final_min_rate,
                "final_mse_mbps2": final_mse,
                "Talloc_ms": final_talloc,
                "cumulative_delay_ms": final_dcum,
                "I1": final["I1"],
                "I2": final["I2"],
                "I3": final["I3"],
            }
        )

    return histories, all_scenario_rows


# ================================================================
# 7. RIS DEPENDENCY SIMULATION
# ================================================================

def simulate_ris_dependency():
    """
    RIS dependency model.

    IMPORTANT:
    This is a service-level abstraction of RIS reconfiguration.
    The current simulation does not use Sionna RT to ray-trace an
    explicit RIS surface.

    t_j,RIS, sleep time and throughput gain are explicit assumptions.
    """

    rows = []

    cumulative_time_ms = 0.0

    # Saturating throughput gain curve.
    # It represents diminishing benefit as more RIS elements/configurations
    # are activated.
    gain_curve = np.array(
        [
            1.05,
            1.12,
            1.20,
            1.27,
            1.32,
            1.35,
            1.37,
            1.38,
        ],
        dtype=np.float64,
    )

    for j, gain in enumerate(gain_curve, start=1):

        t_ris_ms = (
            1.0 + 0.12 * j
        )

        cumulative_time_ms += t_ris_ms

        sleep_ms = max(
            0.5,
            6.0 - 0.45 * j
        )

        rows.append(
            {
                "iteration": j,
                "t_ris_ms": t_ris_ms,
                "T_RIS_ms": cumulative_time_ms,
                "E_gain": float(gain),
                "sleep_ms": sleep_ms,
                "I4": int(
                    cumulative_time_ms > T_PROC_MS
                ),
                "I5": int(
                    sleep_ms < T_SLEEP_MAX_MS
                ),
            }
        )

    Emax = max(
        row["E_gain"]
        for row in rows
    )

    return rows, Emax


# ================================================================
# 8. PLOTS
# ================================================================

def plot_qos_recovery(history):
    x = np.array([
        r["iteration"] for r in history
    ])

    min_rate = np.array([
        r["min_rate_mbps"] for r in history
    ])

    plt.figure(figsize=(8, 5))

    plt.plot(
        x,
        min_rate,
        marker="o",
        label="Minimum user rate",
    )

    plt.axhline(
        np.min(TARGET_MBPS),
        linestyle="--",
        label="Minimum target rate",
    )

    plt.axhline(
        np.min(TARGET_MBPS) - DELTA_Q_MBPS,
        linestyle=":",
        label="Recovery threshold",
    )

    plt.xlabel("Recovery iteration")
    plt.ylabel("Rate (Mbps)")
    plt.title("Sionna-Based QoS Recovery")
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        OUT / "01_qos_recovery.png",
        dpi=250,
    )

    plt.close()


def plot_mse(history):
    x = np.array([
        r["iteration"] for r in history
    ])

    mse = np.array([
        r["mse"] for r in history
    ])

    plt.figure(figsize=(8, 5))

    plt.plot(
        x,
        mse,
        marker="o",
    )

    plt.xlabel("Recovery iteration")
    plt.ylabel("QoS deviation MSE (Mbps²)")
    plt.title("QoS Deviation MSE")
    plt.grid(True, alpha=0.25)
    plt.tight_layout()

    plt.savefig(
        OUT / "02_qos_mse.png",
        dpi=250,
    )

    plt.close()


def plot_talloc(history):
    x = np.array([
        r["iteration"] for r in history
    ])

    y = np.array([
        r["talloc_ms"] for r in history
    ])

    plt.figure(figsize=(8, 5))

    plt.plot(
        x,
        y,
        marker="o",
        label="Cumulative Talloc",
    )

    plt.axhline(
        T_NMIN_MS,
        linestyle="--",
        label="Tnmin",
    )

    plt.xlabel("Recovery iteration")
    plt.ylabel("Cumulative allocation time (ms)")
    plt.title("Talloc and Incident I1")
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        OUT / "03_talloc_I1.png",
        dpi=250,
    )

    plt.close()


def plot_control_delay(history):
    x = np.array([
        r["iteration"] for r in history
    ])

    y = np.array([
        r["d_cum_ms"] for r in history
    ])

    plt.figure(figsize=(8, 5))

    plt.plot(
        x,
        y,
        marker="o",
        label="Cumulative control delay",
    )

    plt.axhline(
        D_MIN_MS,
        linestyle="--",
        label="dmin",
    )

    plt.xlabel("Recovery iteration")
    plt.ylabel("Cumulative delay (ms)")
    plt.title("Control Delay and Incident I2")
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        OUT / "04_control_delay_I2.png",
        dpi=250,
    )

    plt.close()


def plot_qos_error(history):
    x = np.array([
        r["iteration"] for r in history
    ])

    y = np.array([
        r["max_abs_error_mbps"] for r in history
    ])

    plt.figure(figsize=(8, 5))

    plt.plot(
        x,
        y,
        marker="o",
        label="Maximum |eΔQ|",
    )

    plt.axhline(
        E_TH_MBPS,
        linestyle="--",
        label="eth",
    )

    plt.xlabel("Recovery iteration")
    plt.ylabel("Maximum |eΔQ| (Mbps)")
    plt.title("QoS Deviation and Incident I3")
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        OUT / "05_qos_error_I3.png",
        dpi=250,
    )

    plt.close()


def plot_ris_tris(ris_rows):
    x = np.array([
        r["iteration"] for r in ris_rows
    ])

    y = np.array([
        r["T_RIS_ms"] for r in ris_rows
    ])

    plt.figure(figsize=(8, 5))

    plt.plot(
        x,
        y,
        marker="o",
        label="Cumulative TRIS",
    )

    plt.axhline(
        T_PROC_MS,
        linestyle="--",
        label="Tproc",
    )

    plt.xlabel("RIS iteration")
    plt.ylabel("Cumulative TRIS (ms)")
    plt.title("RIS Reconfiguration Time and Incident I4")
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        OUT / "06_ris_TRIS_I4.png",
        dpi=250,
    )

    plt.close()


def plot_ris_emax(ris_rows):
    x = np.array([
        r["iteration"] for r in ris_rows
    ])

    y = np.array([
        r["E_gain"] for r in ris_rows
    ])

    emax = np.max(y)

    plt.figure(figsize=(8, 5))

    plt.plot(
        x,
        y,
        marker="o",
        label="RIS throughput gain",
    )

    plt.axhline(
        emax,
        linestyle="--",
        label="Emax",
    )

    plt.xlabel("RIS iteration")
    plt.ylabel("Relative throughput gain")
    plt.title("RIS Throughput Gain and Emax")
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        OUT / "07_ris_Emax.png",
        dpi=250,
    )

    plt.close()


def plot_scenario_comparison(scenario_rows):
    names = [
        r["scenario"]
        for r in scenario_rows
    ]

    mse = np.array([
        r["final_mse_mbps2"]
        for r in scenario_rows
    ])

    plt.figure(figsize=(10, 5))

    x = np.arange(len(names))

    plt.bar(
        x,
        mse,
    )

    plt.xticks(
        x,
        names,
        rotation=20,
        ha="right",
    )

    plt.ylabel("Final QoS deviation MSE (Mbps²)")
    plt.title("Rservice QoS Performance Across Scenarios")
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()

    plt.savefig(
        OUT / "08_scenario_comparison.png",
        dpi=250,
    )

    plt.close()


def plot_final_user_rates(history):
    final = history[-1]

    users = np.arange(
        1,
        NUM_USERS + 1,
    )

    rates = final["rates"]

    plt.figure(figsize=(9, 5))

    plt.plot(
        users,
        rates,
        marker="o",
        label="Sionna-derived rate",
    )

    plt.plot(
        users,
        TARGET_MBPS,
        marker="x",
        linestyle="--",
        label="Target rate",
    )

    plt.xlabel("User")
    plt.ylabel("Rate (Mbps)")
    plt.title("Final Per-User QoS")
    plt.xticks(users)
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        OUT / "09_final_user_rates.png",
        dpi=250,
    )

    plt.close()


def plot_channel_gains(gains):
    users = np.arange(
        1,
        NUM_USERS + 1,
    )

    plt.figure(figsize=(9, 5))

    plt.semilogy(
        users,
        gains,
        marker="o",
    )

    plt.xlabel("User")
    plt.ylabel("Effective Sionna channel power gain")
    plt.title("3GPP UMi Channel Gains")
    plt.xticks(users)
    plt.grid(True, which="both", alpha=0.25)
    plt.tight_layout()

    plt.savefig(
        OUT / "10_channel_gains.png",
        dpi=250,
    )

    plt.close()


# ================================================================
# 9. MAIN
# ================================================================

def main():

    start_time = time.time()

    print_header("SIONNA R_SERVICE SIMULATION")

    print("Python:", sys.version.split()[0])
    print("Platform:", platform.platform())
    print("PyTorch:", torch.__version__)
    print("Sionna:", getattr(sionna, "__version__", "unknown"))
    print("Device:", DEVICE)
    print("Users:", NUM_USERS)
    print("Total PRBs:", TOTAL_PRBS)
    print("Carrier frequency:", CARRIER_FREQUENCY_HZ / 1e9, "GHz")

    # ------------------------------------------------------------
    # Sionna channel
    # ------------------------------------------------------------

    print_header("1. BUILDING SIONNA 3GPP UMi CHANNEL")

    channel_model, topology = build_umi_channel()

    print("Topology created successfully.")

    # ------------------------------------------------------------
    # Channel realization
    # ------------------------------------------------------------

    print_header("2. SAMPLING SIONNA CHANNEL")

    gains = sample_channel_gains(
        channel_model
    )

    print("\nEffective Sionna channel gains:")

    for user, gain in enumerate(gains, start=1):
        print(
            f"  U{user}: {gain:.6e}"
        )

    # ------------------------------------------------------------
    # Initial allocation
    # ------------------------------------------------------------

    initial_prbs = initial_priority_allocation()

    print("\nInitial PRB allocation:")

    for user, p in enumerate(initial_prbs, start=1):
        priority = (
            "HIGH"
            if user - 1 in HIGH_PRIORITY
            else "LOW"
        )

        print(
            f"  U{user}: {p:2d} PRBs ({priority})"
        )

    print(
        "Total:",
        initial_prbs.sum(),
        "PRBs"
    )

    # ------------------------------------------------------------
    # Scenario simulations
    # ------------------------------------------------------------

    print_header("3. RUNNING SERVICE SCENARIOS")

    histories, scenario_rows = run_scenarios(
        gains
    )

    # ------------------------------------------------------------
    # Select moderate degradation scenario for detailed metric plots
    # ------------------------------------------------------------

    detailed_name = "Moderate degradation"

    detailed_history = histories[
        detailed_name
    ]

    # ------------------------------------------------------------
    # RIS
    # ------------------------------------------------------------

    print_header("4. RUNNING RIS SERVICE DEPENDENCY")

    ris_rows, Emax = simulate_ris_dependency()

    # ------------------------------------------------------------
    # Save detailed histories
    # ------------------------------------------------------------

    save_rows_csv(
        "recovery_history.csv",
        detailed_history,
    )

    save_rows_csv(
        "scenario_summary.csv",
        scenario_rows,
    )

    save_rows_csv(
        "ris_history.csv",
        ris_rows,
    )

    # ------------------------------------------------------------
    # Generate plots
    # ------------------------------------------------------------

    print_header("5. GENERATING FIGURES")

    plot_qos_recovery(
        detailed_history
    )

    plot_mse(
        detailed_history
    )

    plot_talloc(
        detailed_history
    )

    plot_control_delay(
        detailed_history
    )

    plot_qos_error(
        detailed_history
    )

    plot_ris_tris(
        ris_rows
    )

    plot_ris_emax(
        ris_rows
    )

    plot_scenario_comparison(
        scenario_rows
    )

    plot_final_user_rates(
        detailed_history
    )

    plot_channel_gains(
        gains
    )

    # ------------------------------------------------------------
    # Summary
    # ------------------------------------------------------------

    final = detailed_history[-1]

    summary_rows = [
        ("device", DEVICE),
        ("carrier_frequency_GHz",
         CARRIER_FREQUENCY_HZ / 1e9),
        ("num_users", NUM_USERS),
        ("total_PRBs", TOTAL_PRBS),
        ("selected_detailed_scenario",
         detailed_name),
        ("recovery_iterations",
         len(detailed_history)),
        ("final_min_rate_Mbps",
         float(np.min(final["rates"]))),
        ("final_MSE_Mbps2",
         float(final["mse"])),
        ("final_Talloc_ms",
         float(final["talloc_ms"])),
        ("final_cumulative_control_delay_ms",
         float(final["d_cum_ms"])),
        ("I1", final["I1"]),
        ("I2", final["I2"]),
        ("I3", final["I3"]),
        ("final_TRIS_ms",
         float(ris_rows[-1]["T_RIS_ms"])),
        ("Emax",
         float(Emax)),
        ("I4", ris_rows[-1]["I4"]),
        ("I5", ris_rows[-1]["I5"]),
        ("runtime_seconds",
         time.time() - start_time),
    ]

    save_summary(
        summary_rows
    )

    # ------------------------------------------------------------
    # Console results
    # ------------------------------------------------------------

    print_header("6. FINAL RESULTS")

    print(
        f"Detailed scenario: {detailed_name}"
    )

    print(
        f"Recovery iterations : "
        f"{len(detailed_history)}"
    )

    print(
        f"Final minimum rate  : "
        f"{np.min(final['rates']):.3f} Mbps"
    )

    print(
        f"Final QoS MSE       : "
        f"{final['mse']:.3f} Mbps^2"
    )

    print(
        f"Talloc              : "
        f"{final['talloc_ms']:.3f} ms"
    )

    print(
        f"Cumulative d_i      : "
        f"{final['d_cum_ms']:.3f} ms"
    )

    print(
        f"I1={final['I1']}  "
        f"I2={final['I2']}  "
        f"I3={final['I3']}"
    )

    print(
        f"TRIS                : "
        f"{ris_rows[-1]['T_RIS_ms']:.3f} ms"
    )

    print(
        f"Emax                : "
        f"{Emax:.3f}"
    )

    print(
        f"I4={ris_rows[-1]['I4']}  "
        f"I5={ris_rows[-1]['I5']}"
    )

    print(
        "\nAll results saved in:",
        OUT.resolve()
    )

    print("\nGenerated files:")

    for path in sorted(OUT.iterdir()):
        print("  ", path.name)


if __name__ == "__main__":
    main()



SIONNA R_SERVICE SIMULATION
Python: 3.13.15
Platform: Linux-6.6.122+-x86_64-with-glibc2.39
PyTorch: 2.11.0+cpu
Sionna: 2.1.0
Device: cpu
Users: 6
Total PRBs: 24
Carrier frequency: 3.5 GHz

1. BUILDING SIONNA 3GPP UMi CHANNEL
Topology created successfully.

2. SAMPLING SIONNA CHANNEL

Effective Sionna channel gains:
  U1: 5.414179e-10
  U2: 8.732951e-15
  U3: 2.338522e-08
  U4: 1.035071e-12
  U5: 2.557256e-12
  U6: 9.769670e-13

Initial PRB allocation:
  U1:  6 PRBs (HIGH)
  U2:  6 PRBs (HIGH)
  U3:  6 PRBs (HIGH)
  U4:  2 PRBs (LOW)
  U5:  2 PRBs (LOW)
  U6:  2 PRBs (LOW)
Total: 24 PRBs

3. RUNNING SERVICE SCENARIOS

Running scenario: Normal

Running scenario: Moderate degradation

Running scenario: Severe degradation

Running scenario: Severe degradation + control delay

4. RUNNING RIS SERVICE DEPENDENCY

5. GENERATING FIGURES

6. FINAL RESULTS
Detailed scenario: Moderate degradation
Recovery iterations : 12
Final minimum rate  : 0.329 Mbps
Final QoS MSE       : 45.118 Mbps^2
Talloc 